# 01 · NDVI time series: olive groves vs arable land (Martos, Jaén)

**Goal.** See with real data why olive groves are hard to monitor from space: their NDVI should stay
almost flat all year, while arable land (mostly cereal) peaks in spring and drops after harvest.

**Concepts used here**
- **NDVI** = (NIR − Red) / (NIR + Red), from Sentinel-2 bands **B08** (near infrared) and **B04** (red).
  Dense green vegetation ≈ 0.8, olive grove with bare soil between trees ≈ 0.3–0.5, bare soil ≈ 0.1–0.2.
- **SCL** (Scene Classification Layer): a Sentinel-2 L2A band that labels every pixel (cloud, shadow,
  vegetation, soil…). We keep only *vegetation* and *bare soil* pixels.
- **Mixed pixels**: a 10 m pixel on a parcel border mixes the crop with roads or neighbours, so we
  shrink every parcel 10 m inwards before averaging.

**Copernicus quota.** Everything heavy runs on the server (masking, NDVI, mean per parcel); we download
a small CSV from **one batch job**, cached under a hash of the request. Re-running this notebook with the
same parameters costs **0 credits**. Credits used are logged in `data/cache/openeo/credits_log.csv`.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import openeo
import pandas as pd

from oliveye.pipeline.openeo_ndvi import (
    build_ndvi_timeseries,
    load_ndvi_csv,
    prepare_geometries,
    run_cached_batch_job,
    select_parcels,
    spatial_extent,
)
from oliveye.pipeline.sigpac import subset_path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SIGPAC_DIR = REPO_ROOT / "data" / "raw" / "sigpac"
CACHE_DIR = REPO_ROOT / "data" / "cache" / "openeo"

PROVINCE, MUNICIPALITY = "23", "60"  # Jaén, Martos
START_DATE, END_DATE = "2024-01-01", "2026-01-01"  # two full years (end date is exclusive)

# Colors: first two slots of a colorblind-validated categorical palette.
COLORS = {"OV": "#2a78d6", "TA": "#eb6834"}
LABELS = {"OV": "Olive grove (OV)", "TA": "Arable land (TA)"}

## 1. Load the SIGPAC parcels of Martos
Created by `scripts/fetch_sigpac_subset.py`. Geometries are in EPSG:25830 (metres).

In [ ]:
parcels = gpd.read_parquet(subset_path(SIGPAC_DIR, PROVINCE, MUNICIPALITY))
print("CRS:", parcels.crs.to_string())
parcels.groupby("land_use_code")["area_ha"].describe().round(2)

## 2. Pick a small, comparable sample

Arable land is scarce in Martos, so we pick **2 arable parcels first** and then **5 olive parcels within
2 km of them**. Keeping all parcels close together means similar weather and a tiny bounding box, which is
what openEO bills for.

In [ ]:
ta = select_parcels(parcels, "TA", n=2, min_area_ha=1.0)
ov = select_parcels(parcels, "OV", n=5, min_area_ha=3.0, near=ta, max_distance_m=2_000)
sample = pd.concat([ta, ov])
geometries = prepare_geometries(sample, inner_buffer_m=10)

geometries.drop(columns="geometry")

In [ ]:
extent = spatial_extent(geometries)
bbox_km2 = gpd.GeoSeries.from_xy([extent["west"], extent["east"]], [extent["south"], extent["north"]],
                                 crs="EPSG:4326").to_crs("EPSG:25830").total_bounds
width_km, height_km = (bbox_km2[2] - bbox_km2[0]) / 1e3, (bbox_km2[3] - bbox_km2[1]) / 1e3
print(f"Bounding box sent to openEO: {width_km:.1f} km x {height_km:.1f} km = {width_km * height_km:.1f} km²")

fig, ax = plt.subplots(figsize=(6, 6))
for code_, group in geometries.to_crs("EPSG:25830").groupby("land_use_code"):
    group.plot(ax=ax, color=COLORS[code_], label=LABELS[code_])
ax.set_title("Sample parcels (shrunk 10 m inwards)")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=COLORS[c]) for c in COLORS],
          labels=list(LABELS.values()), frameon=False)
ax.set_axis_off()
plt.show()

## 3. Build the openEO request and run it (once)

Connecting uses your saved refresh token and is free. The batch job is only created if the result is
**not** already in the cache. Expect a few minutes of waiting the first time.

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

process = build_ndvi_timeseries(geometries, START_DATE, END_DATE, connection=connection)
csv_path = run_cached_batch_job(process, CACHE_DIR, title="oliveye-01-ndvi-martos-2024-2025")

In [ ]:
ndvi = load_ndvi_csv(csv_path, geometries)
print(f"{len(ndvi)} cloud-free observations")
ndvi.groupby(["land_use_code", "parcel_id"])["ndvi"].agg(["count", "mean", "min", "max"]).round(3)

## 4. Look at the curves

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
for (code_, _parcel), series in ndvi.groupby(["land_use_code", "parcel_id"]):
    ax.plot(series["date"], series["ndvi"], color=COLORS[code_], linewidth=1, alpha=0.45)

monthly = (ndvi.set_index("date").groupby("land_use_code")["ndvi"]
           .resample("MS").median().reset_index())
for code_, series in monthly.groupby("land_use_code"):
    mid_month = series["date"] + pd.Timedelta(days=14)  # plot each median at mid-month
    ax.plot(mid_month, series["ndvi"], color=COLORS[code_], linewidth=2.5, label=LABELS[code_])
    ax.annotate(LABELS[code_], (mid_month.iloc[-1], series["ndvi"].iloc[-1]), xytext=(6, 0),
                textcoords="offset points", va="center", color="#333333")

ax.set_title("NDVI per parcel (thin) and monthly median (thick) — Martos, 2024–2025")
ax.set_ylabel("NDVI")
ax.set_ylim(0, 1)
ax.grid(axis="y", color="#e5e5e5")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Same data as a table: median NDVI by calendar month, to compare the seasonal shape.
seasonal = (ndvi.assign(month=ndvi["date"].dt.month)
            .pivot_table(index="month", columns="land_use_code", values="ndvi", aggfunc="median"))
seasonal.round(2)

## 5. What to look for (write your conclusions here)

- Does arable land show a clear spring peak (Mar–Apr) and a drop after harvest (Jun–Jul)?
- How flat is the olive curve? What is its yearly range compared with arable land?
- Is there a summer dip in olive NDVI (water stress) and a recovery after autumn rains?
- How many cloud-free dates did we get per parcel, and in which months are there gaps?

**Credits:** check `data/cache/openeo/credits_log.csv` and the Billing page, and note how much this job cost.
That number tells us how to budget Phase 2 (thousands of parcels).